# Digital Forge Reel — Colab Renderer

Render the HTML/CSS/JS scene into a vertical MP4 video directly on Google Colab.

**Recommended runtime:** GPU (Runtime → Change runtime type → T4 GPU)

## What this notebook does
1. Clones/uploads the project
2. Installs Node, Chromium, ffmpeg, Python deps
3. Renders the English + Arabic videos
4. Downloads them to your machine

## Step 1 — Upload the project zip

Upload `digital-forge-reel.zip` (the zip file you received) to Colab's `/content/` directory using the file browser on the left.

Then run the cell below to unzip it.

In [ ]:
# Unzip the project
%cd /content
!unzip -o digital-forge-reel.zip -d /content/
%cd /content/digital-forge-reel
!ls -la

## Step 2 — Install Node.js + dependencies

Colab doesn't ship with Node by default. This cell installs Node 20 + all system deps (Chromium, fonts, ffmpeg, NVIDIA tools if GPU runtime).

In [ ]:
# Install Node.js 20
!curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash -
!sudo apt-get install -y nodejs
!node --version && npm --version

In [ ]:
# Install all project dependencies (Chromium, fonts, ffmpeg, Python deps, Playwright browser)
# This takes ~3-5 minutes the first time.
!node bin/forge-setup --verbose

## Step 3 — Verify GPU availability (optional)

If you're on a GPU runtime, this shows your GPU. NVENC encoding will be auto-selected.

In [ ]:
!nvidia-smi 2>/dev/null || echo 'No NVIDIA GPU — will use CPU encoder (slower but works)'
!ffmpeg -hide_banner -encoders | grep -E 'h264_nvenc|h264_vaapi|h264_qsv|h264_videotoolbox|libx264' | head -10

## Step 4 — Render the English video

Renders at **full 1080×1920 @ 60fps**. With GPU capture + NVENC encoding, this takes ~10-15 minutes for a 30s reel.

On CPU-only runtimes, use `--scale 0.5` for ~4x faster render (540×960 source, upscaled to 1080×1920).

In [ ]:
# Full quality GPU render (Colab T4 GPU runtime)
!node bin/forge-render scenes/digital-forge-reel-en.html \
    --output output/digital-forge-reel-en.mp4 \
    --music audio/forge_theme.wav \
    --gpu \
    --encoder auto \
    --fps 30 \
    --target-fps 60 \
    --scale 1.0 \
    --interpolate dup \
    --log-level info

In [ ]:
# Fast CPU render (no GPU needed) — half-res capture, ~5 min
# !node bin/forge-render scenes/digital-forge-reel-en.html \
#     --output output/digital-forge-reel-en-fast.mp4 \
#     --music audio/forge_theme.wav \
#     --scale 0.5 \
#     --fps 15 \
#     --target-fps 60 \
#     --interpolate dup

## Step 5 — Render the Arabic video (Algerian Darija)

In [ ]:
!node bin/forge-render scenes/digital-forge-reel-ar.html \
    --output output/digital-forge-reel-ar.mp4 \
    --music audio/forge_theme.wav \
    --gpu \
    --encoder auto \
    --fps 30 \
    --target-fps 60 \
    --scale 1.0 \
    --interpolate dup \
    --log-level info

## Step 6 — Download the videos

Saves to your Google Drive (persistent) or downloads to your machine.

In [ ]:
# Option A: Save to Google Drive (persistent across sessions)
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/digital-forge-reel
!cp output/*.mp4 /content/drive/MyDrive/digital-forge-reel/
!ls -lh /content/drive/MyDrive/digital-forge-reel/

In [ ]:
# Option B: Download directly to your machine
from google.colab import files
import os

for f in os.listdir('output'):
    if f.endswith('.mp4'):
        print(f'Downloading {f}...')
        files.download(f'output/{f}')

## Troubleshooting

**"Chromium not found"** → Run `node bin/forge-setup` again. On Colab it installs `chromium-browser` via apt.

**"ffmpeg lacks h264_nvenc"** → Make sure you're on a GPU runtime. Switch via Runtime → Change runtime type → T4 GPU, then re-run Step 2.

**Video is offset / has black bars** → Make sure you're using the latest project version. The renderer now uses viewport-aligned CSS instead of transform-based scaling.

**Out of memory** → Use `--scale 0.5` to capture at half resolution, or `--fps 15` to halve the frame count.

**Render too slow** → Add `--gpu` flag (requires GPU runtime) and use `--encoder nvenc`. Expect 5-10x speedup.

**Want to modify the scene** → Edit `scenes/digital-forge-reel-en.html` (or `-ar.html`) and re-run the render cell. The HTML is self-contained — no build step.